# 1. Importacion de librerias
Se importan las libreias necesarias

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from copy import deepcopy
from pathlib import Path

# ----------------------------

import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import torch

from keras import layers
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

print(f"GPU disponible: {'Si' if torch.cuda.is_available() else 'No'}")


GPU disponible: Si


# 2. Funciones y clases necesarias para el AG

Cromosoma: [alpha,  batch, phi, rho]



### Se colocan las constantes de los anelos permitidos por gen

Rangos:
-   alpha = [1e-4, 1e-2]
-   batch = {8, 16, 32, 64}
-   phi = {adam, RMSprop, SGD}
-   rho = [0.0, 0.5]

In [1]:
ALPHA = (1e-4, 1e-2)
BATCH = (8, 16, 32, 64)
PHI = ("adam", "sgd", "rmsprop")
RHO = (0.0, 0.50)

### Se definen constantes utiles para las funciones

In [ ]:
# Tipo de cruzamiento por hiperparámetro:
UNIFORM_CROSS = ("batch", "phi")
BLEND_CROSSOVER = ("alpha", "rho")

# Probabilidad de mutacion:
MUTATION_RATE = 0.15

# Generaciones maximas
GEN_MAX = 80

# Paciencia en las generaciones
GEN_PAT = 10

# Individuos base en la seleccion
K_SELECT = 4

# Inicializacion con semilla
SEED = 29
random.seed(SEED)

### Definicion de la clase del cromosoma
Se crea una clase la cual contendra los hiperparametros del entrenamiento

In [ ]:
class HyperParams:
    def __init__(
        self,
        alpha: float|None = None,
        batch: int|None = None,
        phi: str|None = None,
        rho: float|None = None
    ):
        self.alpha = alpha
        self.batch = batch
        self.phi = phi
        self.rho = rho

    def __getitem__(
        self, 
        key: str,
    ):
        return getattr(self, key)
    
    def __setitem__(
        self, 
        key: str, 
        value,
    ):
        setattr(self, key, value)

    def keys(
        self,
    ):
        return ["alpha", "batch", "phi", "rho"]

    def copy(
        self,
    ):
        return deepcopy(self)


### Definicion de la estrutura del individuo
Se implementa una clase para cada invidiuo, la cual guarde su cromosoma, fitness y tenga metodos para el cruzamiento y mutaciones.

In [ ]:
class Individual:
    def __init__(
        self,
        chromosome: HyperParams|None = None,
    ):
        if chromosome is not None:
            self.chromosome = chromosome.copy()
        else:
            raise ValueError("Error: chromosome es nulo")
        
        self.fitness: float = -1

    def crossover(
        self,
        other,
    ):
        chromosome1 = HyperParams()
        chromosome2 = HyperParams()

        # Se hace un cruzamiento uniforme para los genes phi y batch:
        for param in UNIFORM_CROSS:
            if random.uniform(0, 1) <= 0.5:
                chromosome1[param] = self.chromosome[param]
                chromosome2[param] = other.chromosome[param]
            else:
                chromosome1[param] = other.chromosome[param]
                chromosome2[param] = self.chromosome[param]

        # Se hace un cruzamiento blend para los genes alpha y rho:
        for param in BLEND_CROSSOVER:
            alpha = random.uniform(0, 1)
            chromosome1[param] = alpha * self.chromosome[param] + (1 - alpha) * other.chromosome[param]
            chromosome2[param] = alpha * other.chromosome[param] + (1 - alpha) * self.chromosome[param]

        return Individual(chromosome1), Individual(chromosome2)
    
    def mutation(
        self,
    ):
        chromosome = self.chromosome.copy()

        for param in chromosome.keys():
            if random.uniform(0, 1) <= MUTATION_RATE:
                if param == "alpha":
                    chromosome[param] = random.uniform(*ALPHA)
                elif param == "batch":
                    chromosome[param] = random.choice(BATCH)
                elif param == "phi":
                    chromosome[param] = random.choice(PHI)
                elif param == "rho":
                    chromosome[param] = random.uniform(*RHO)

        return Individual(chromosome)

### Funcion para obtener el fitness de un cromosoma

In [ ]:
def get_fitness(
    chromosome: HyperParams,

) -> float:
    #TODO xd
    # Despues de tener el entrenamiento de las cnn

### Funcion para evaluar una poblacion de individuos

In [ ]:
def eval_population(
    population: list[Individual],
):
    pop_size = len(population)

    for i in range(pop_size):
        if population[i].fitness == -1:
            population[i].fitness = get_fitness(population[i].chromosome)

### Funcion para inicializar una poblacion de individuos


In [ ]:
def init_population(
    size: int, 
):
    population = []
    chromosome_already_used = set()
    for _ in range(size):
        alpha = random.uniform(*ALPHA)
        batch = random.choice(BATCH)
        phi = random.choice(PHI)
        rho = random.uniform(*RHO)

        chromosome = (alpha, batch, phi, rho)
        population.append(Individual(HyperParams(*chromosome)))
    
    return population


### Funcion para la seleccion de padres

In [ ]:
def select_parents(
    population: list[Individual],
    tournament_size: int,
):
    list_indiv = []

    x1 = np.random.permutation(len(population))
    y1 = x1[:tournament_size]

    for i in range(tournament_size):
        list_indiv.append(population[y1[i]].fitness)

    iParent1 = np.argmax(list_indiv)

    list_indiv = []
    x2 = np.delete(x1, iParent1)
    x2 = np.random.permutation(x2)
    y2 = x2[:tournament_size]

    for i in range(tournament_size):
        list_indiv.append(population[y2[i]].fitness)

    iParent2 = np.argmax(list_indiv)

    return population[y1[iParent1]], population[y2[iParent2]]

### Funcion para la seleccion de la poblacion

In [ ]:
def select_survivors(
    population: list[Individual],
    offspring: list[Individual],
    num_survivors: int,
):
    next_population = []
    population.extend(offspring)
    population.sort(key=lambda x: x.fitness, reverse=True)

    next_population = population[:num_survivors]
    
    return next_population

    

### Algoritmo genetico para encontrar soluciones

In [ ]:
def genetic_algo(
    population,
    n_gen = GEN_MAX,
    p_gen = GEN_PAT,
):
    
    pop_size = len(population)

    eval_population(population)

    best = sorted(population, key=lambda x: x.fitness, reverse=True)[0]
    best_fitness = [best.fitness]

    print(f"Poblacion inicial, mejor fitness = {best_fitness[:1]}")

    for gen in range(n_gen):

        mating_pool = []

        for i in range(int(pop_size/2)):
            mating_pool.append(select_parents(
                population=population,
                tournament_size=K_SELECT,
            ))
        
        offspring = []
        
        for i in range(int(pop_size/2)):
            papa = mating_pool[i][0]
            mama = mating_pool[i][1]
            offspring.extend(papa.crossover(
                other=mama
            ))

        offspring = [child.mutation() for child in offspring]

        eval_population(
            population=offspring
        )

        population = select_survivors(
            population=population,
            offspring=offspring,
            num_survivors=pop_size,
        )

        best = sorted(population, key=lambda x: x.fitness, reverse=True)[0]
        best_fitness.append(best.fitness)

        if best_fitness[-1] > best_fitness[-2]:
            print(f"Generación {gen}, mejor fitness = {best_fitness[-1]}")

    print(f"Mejor individuo en la ultima geneacion con fitness = {best_fitness[-1]}")
    return best, best_fitness

     

# 3. Funciones y clases necesarias para el Entrenamiento de CNN


In [ ]:
N_CLASS = 3

OPTIMIZERS = {
    "adam": keras.optimizers.Adam,
    "sgd": keras.optimizers.SGD,
    "rmsprop": keras.optimizers.RMSprop,
}

MODELS = {
    "MobileNetV2": {
        "fn": keras.applications.MobileNetV2,
        "preprocess": keras.applications.mobilenet_v2.preprocess_input,
    },
    "EfficientNetB0": {
        "fn": keras.applications.EfficientNetB0,
        "preprocess": keras.applications.efficientnet.preprocess_input,
    },
}



### Funcion para generar un modelo

In [ ]:
def build_model(
    backbone: str,
    hiperparams: HyperParams,
):
    

    base = MODELS[backbone]["fn"](
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = False

    data_aug = keras.Sequential(
        [
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.05),
        ],
        name="augment"
    )

    inputs = keras.Input(shape=(224, 224, 3))
    x = data_aug(inputs)
    x = layers.Lambda(MODELS[backbone]["preprocess"])(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(hiperparams["rho"])(x)
    outputs = layers.Dense(N_CLASS, activation="softmax")(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=OPTIMIZERS[hiperparams["phi"]](learning_rate=hiperparams["alpha"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model

# 4. Carga de dataset y division


In [ ]:
build_model("mobilenetv2", HyperParams(1e-3, 32, "adam", 0.3)).summary()

### Se define las constantes necesarias para este apartado

In [ ]:
# Direccion a buscar
DIR_DATASET = Path("data/PlantVillage")

CLASS_NAMES = sorted([dir.name for dir in DIR_DATASET.iterdir()])
N_CLASS = len(CLASS_NAMES)

print("="*60)
print("Clases presentes: ")
[print(f"   - {class_name}") for class_name in CLASS_NAMES]
print("="*60)

### Funcion para la carga del dataset

In [ ]:
def load_images(
    data_dir: Path,
):
    X, y = [], []

    for idx, cname in enumerate(CLASS_NAMES):
        for p in (data_dir / cname).glob("*.JPG"):
            img = Image.open(p).convert("RGB")
            X.append(np.asarray(img, dtype=np.uint8))
            y.append(idx)
    
    return np.stack(X), np.array(y)


In [ ]:
# Visualizacion del total de imagenes
X_raw, y_all = load_images(
    DIR_DATASET,
)

print("="*60)
print(f"Total: {X_raw.shape}")
print("="*60)

plt.figure(figsize=(3 * N_CLASS, 3))
for idx, cname in enumerate(CLASS_NAMES):
    i = np.where(y_all == idx)[0][0]                 
    plt.subplot(1, N_CLASS, idx + 1)
    plt.imshow(X_raw[i].astype("uint8"))
    plt.title(f"{cname}\n(n={np.sum(y_all == idx)})")
    plt.axis("off")
plt.tight_layout()
plt.show()

# E1 Funciones y clases necesarias para el Entrenamiento de CNN
